In [ ]:
# @title MMMv13 - model spec with differentiated video priors - takes about 8min to run (a100)

# =============================================================================
# CHANGES FROM v12:
# - non_media_cols: removed Conversions_BasketSize and Conversions_Pricing (confounding the fit; capturing demand-side
#   variation media should explain).
# - roi_mu for video: was [1.65 x 5] (uniform prior). Now per-partner, tiered by Ken's outside view of inventory type:
#     * Performance / demand-gen tier (high prior ROAS):  Google, MNTN, Epsilon
#     * Branding / awareness tier      (mid prior ROAS):   Hulu, Paramount
#   Paramount later bumped from 1.20 -> 1.40 after realized history showed consistent 4.6-6.2x ROAS.
# - roi_sigma for video: 0.25 -> 0.50 so 156 weeks of national data can move the posteriors off the prior. The old
#   0.25 was so tight that v12 video posteriors all collapsed to ~5.3-5.7x (basically still on the prior).
# - ec_sigma for video: 0.30 -> 0.50 to try to identify per-partner Hill saturation points. Didn't pan out empirically
#   (all video ec_m posteriors stayed near prior mean of 1.0), but no harm leaving loose.
# =============================================================================

# =============================================================================
# KNOWN LIMITATION - Google low-spend ROAS extrapolation:
# At low Google spend (~$9k/quarter) the model produces implausible aligned ROAS (20-30x), while at high spend
# ($217k in 2025Q4) it produces a credible 2.3x. The high-spend point is trustworthy; the low-spend reading is
# the model's Hill curve passing through near-zero data, NOT a measurement.
#
# Why this happens: Hill with slope=1 has marginal response = 1/ec at x=0. To fit both the high-spend saturation
# and the low-spend revenue, the model picks low ec and high beta - which inflates the low-spend extrapolation.
# No prior tweak cleanly fixes this without distorting fit elsewhere; the issue is the functional form.
#
# How to interpret: trust Google's ROAS at spend levels NEAR what it actually ran (~$10k-$220k/qtr based on history).
# Don't trust extrapolations far below that range. The scenario planner will recommend cutting Google to ~$10k/qtr
# at 30x - do NOT take that recommendation at face value. The real fix is a YouTube holdout / incrementality test
# for prior calibration on roi_m[Video_Google].
# =============================================================================

# 1) Install dependencies (colab + scenarioplanner + cuda - all in one pip call)
!pip -q install --upgrade google-meridian[colab,scenarioplanner,and-cuda]

# 2) Authenticate BigQuery + Drive + Sheets in one popup; mount Drive
from google.colab import auth, drive
import datetime

credentials = auth.authenticate_user([
    'https://www.googleapis.com/auth/bigquery',
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets',
])
drive.mount('/content/drive')

# 3) Imports
import arviz as az
import numpy as np
import pandas as pd
import pandas_gbq
from meridian import constants
from meridian.analysis import optimizer as budget_optimizer_module
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.data import data_frame_input_data_builder as builder_module
from meridian.model import model as mmm_module
from meridian.model import prior_distribution
from meridian.model import spec
import tensorflow as tf
import tensorflow_probability as tfp

# 4) Runtime checks
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)
print("TensorFlow version:", tf.__version__)

# 5) Pull data from BigQuery. abc.mmm is the weekly blender view (media + promos + controls);
#    'In Model' filter excludes weeks we don't want to fit (burn-in, hold-outs).
project_id = 'donut-426'
view_sql = """SELECT * FROM abc.mmm WHERE Model_Dates = 'In Model'"""

df_bq = pandas_gbq.read_gbq(view_sql, project_id=project_id)
if 'time' not in df_bq.columns:
    raise ValueError("Expected a 'time' column in BigQuery data.")
df_bq['time'] = pd.to_datetime(df_bq['time'])

# 6) Channel / variable definitions. CRITICAL: media_channels order must match media_impression_cols and
#    media_spend_cols index-for-index, AND must match the vectorized priors below (roi_mu[i] applies to
#    media_channels[i]). Don't reorder these lists without also reordering the prior tensors.
media_channels = ['Meta', 'Search', 'PMAX', 'Amex',
                  'Video_Epsilon', 'Video_Google', 'Video_Hulu',
                  'Video_MNTN', 'Video_Paramount']

# Per-channel media-activity metric. Note Search uses CLICKS, everything else (including Amex) uses IMPRESSIONS.
# This list is parallel to media_channels.
media_impression_cols = ['Meta_impression', 'Search_click', 'PMAX_impression', 'Amex_impression',
                         'Video_Epsilon_impression', 'Video_Google_impression', 'Video_Hulu_impression',
                         'Video_MNTN_impression', 'Video_Paramount_impression']

media_spend_cols = [f"{ch}_spend" for ch in media_channels]

# Non-media treatments enter the model linearly (no Hill/adstock). v13: removed Conversions_BasketSize and
# Conversions_Pricing - they were confounding the fit (capturing demand-side variation media should explain).
non_media_cols = ['Vault_Drops','Email_sends','Redemptions_Other','StoreCount']

# Controls also enter linearly. Redemptions split into Birthday/Signup vs Other so the structurally-elevated
# Birthday signup baseline doesn't wash out general redemption signal.
control_cols = ['Category_Interest', 'Hurricanes', 'LightningSales', 'Redemptions_Birthday_Signup','Celebs']

population_col = 'population' if 'population' in df_bq.columns else None

required_cols = (['time', 'geo', 'Conversions_Revenue']
                 + media_impression_cols + media_spend_cols
                 + non_media_cols + control_cols)
missing = [c for c in required_cols if c not in df_bq.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Verify your BigQuery schema.")
print("All key columns present. Shape:", df_bq.shape)

# 7) Build Meridian InputData. KPI is revenue (continuous, dollars). Builder needs a 'geo' column; for a national
#    model this has a single value ('national_geo') and Meridian warns later that aggregate_geos=True is enforced
#    - that's expected.
builder = builder_module.DataFrameInputDataBuilder(
    kpi_type='revenue',
    default_kpi_column='Conversions_Revenue'
)
builder = builder.with_kpi(df_bq)
builder = builder.with_media(
    df_bq,
    media_cols=media_impression_cols,
    media_spend_cols=media_spend_cols,
    media_channels=media_channels
)
if population_col:
    builder = builder.with_population(df_bq, population_col=population_col)
builder = builder.with_non_media_treatments(df_bq, non_media_treatment_cols=non_media_cols)
builder = builder.with_controls(df_bq, control_cols=control_cols)
input_data = builder.build()
print("Input Data built successfully.")

# 8) PRIORS - v13.
# =============================================================================
# Index for ALL vectorized tensors below (must match media_channels order):
#   0=Meta, 1=Search, 2=PMAX, 3=Amex,
#   4=Video_Epsilon, 5=Video_Google, 6=Video_Hulu, 7=Video_MNTN, 8=Video_Paramount
#
# Reading the ROI priors:
#   roi_mu    = LogNormal 'loc'   (mean in log-space)
#   roi_sigma = LogNormal 'scale' (sd in log-space)
#   prior MEDIAN ROAS = exp(loc);   prior MEAN ROAS = exp(loc + sigma^2/2)
#   95% prior CI factor around the median = exp(+/- 1.96 * sigma)
#
# Video tiers per Ken's outside view:
#   - MNTN & Epsilon: performance CTV, optimized to convert -> high ROAS prior
#   - Google:         demand-gen optimizing store visits, very strong late-2025+. Google's data is concentrated in
#                     that recent period so a bullish prior aligns with the period the model has most signal from.
#                     (See known limitation above re: low-spend extrapolation - prior is right for the run range.)
#   - Hulu:           general branding / awareness -> mid ROAS prior
#   - Paramount:      premium placement, mid-tier between branding and perf. Realized 4.6-6.2x history supports
#                     loc=1.40 (bumped from initial 1.20 branding tag).
# =============================================================================
roi_mu = tf.constant([
    1.40,   # Meta             prior mean ~4.5x
    1.35,   # Search           prior mean ~4.0x
    1.15,   # PMAX             prior mean ~3.4x
    1.15,   # Amex             prior mean ~3.4x
    1.60,   # Video_Epsilon    prior mean ~5.6x   performance CTV  (perf tier)
    1.65,   # Video_Google     prior mean ~5.9x   demand gen / store visits (perf tier)
    1.20,   # Video_Hulu       prior mean ~3.7x   general branding (brand tier)
    1.60,   # Video_MNTN       prior mean ~5.6x   performance CTV  (perf tier)
    1.40,   # Video_Paramount  prior mean ~4.6x   premium placement, mid-tier (was 1.20, realized 4.6-6.2x)
], dtype=tf.float32)

# Video sigma 0.50 (was 0.25 in v12 - too tight, posteriors collapsed onto prior with no differentiation).
# 0.50 -> 95% prior CI roughly [0.4x, 2.7x] around the median: loose enough for data to update, still informed.
roi_sigma = tf.constant([
    0.40,   # Meta
    0.28,   # Search    tight - well-measured channel; 95% mass ~2.2-6.5x
    0.45,   # PMAX
    0.45,   # Amex
    0.50,   # Video_Epsilon
    0.50,   # Video_Google
    0.50,   # Video_Hulu
    0.50,   # Video_MNTN
    0.50,   # Video_Paramount
], dtype=tf.float32)

roi_prior = tfp.distributions.LogNormal(loc=roi_mu, scale=roi_sigma)

# Adstock alpha priors (decay rate per channel).
# - Beta(2, 2) for non-video: symmetric, mean 0.5, mild prior toward moderate decay.
# - Beta(3, 2) for video: mean 0.6, biased toward slightly slower decay - video impressions typically lag
#   conversions more than search/social.
alpha_conc1 = tf.constant([2.0, 2.0, 2.0, 2.0, 3.0, 3.0, 3.0, 3.0, 3.0], dtype=tf.float32)
alpha_conc0 = tf.constant([2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0], dtype=tf.float32)
alpha_prior = tfp.distributions.Beta(concentration1=alpha_conc1, concentration0=alpha_conc0)

# Hill EC50 priors (saturation half-point, on the scaled-media axis). v13 loosened video sigmas from 0.30 -> 0.50
# to try to identify per-partner saturation. In practice the data doesn't have enough cross-spend variation to
# move ec_m off the prior mean of 1.0 - all video posteriors came back at ~1.03 +/- 0.46. No harm leaving loose.
#
# Note on the Google low-spend issue: a naive fix would be to set ec_mu[Google] lower to force earlier saturation,
# but that makes the problem WORSE - Hill marginal at x=0 is 1/ec, so lower ec means STEEPER ramp at zero and
# HIGHER implied low-spend ROAS. Setting ec higher would help with extrapolation but conflicts with observed
# high-spend saturation. The real fix is calibration, not a prior tweak.
ec_mu = tf.constant([1.0] * 9, dtype=tf.float32)
ec_sigma = tf.constant([
    0.5,   # Meta
    0.5,   # Search
    0.5,   # PMAX
    0.5,   # Amex
    0.50,  # Video_Epsilon
    0.50,  # Video_Google
    0.50,  # Video_Hulu
    0.50,  # Video_MNTN
    0.50,  # Video_Paramount
], dtype=tf.float32)
ec_prior = tfp.distributions.TruncatedNormal(loc=ec_mu, scale=ec_sigma, low=0.1, high=10.0)

prior = prior_distribution.PriorDistribution(
    roi_m=roi_prior,
    alpha_m=alpha_prior,
    ec_m=ec_prior,
)
print("v13 priors defined: tiered video means, loosened video sigmas (roi & ec).")
print("Known limitation: Google low-spend ROAS extrapolation is unreliable - trust scale-point readings only.")

# Adstock kernel family per channel.
# - geometric: classic exponential decay weight w_L = (1-alpha) * alpha^L. Peak at lag 0. Use for immediate-
#   response channels (search/social/Amex).
# - binomial: Meridian's two-parameter form allowing ramp-then-decay shapes (peak NOT at lag 0). Use for video
#   where conversions typically don't land the same week as the impression.
adstock_map = {
    "Search":          "geometric",
    "Meta":            "geometric",
    "PMAX":            "geometric",
    "Amex":            "geometric",
    "Video_Epsilon":   "binomial",
    "Video_Google":    "binomial",
    "Video_Hulu":      "binomial",
    "Video_MNTN":      "binomial",
    "Video_Paramount": "binomial",
}

# 9) Model spec.
# - media_effects_dist='log_normal': keeps channel effects positive (no negative ROI draws), matches LogNormal roi prior.
# - max_lag=12 weeks: revenue from a week's media can land up to 12 weeks later. Long enough for video's slow tail;
#   short enough to avoid spurious late-season fits.
# - hill_before_adstock=True: Hill saturation applied per-week, THEN adstock convolved over the saturated values.
#   Makes the adstock kernel shape independent of spend level (clean impulse-response interpretation), and is what
#   the decay-by-channel diagnostic cell relies on.
model_spec = spec.ModelSpec(
    prior=prior,
    media_effects_dist='log_normal',
    adstock_decay_spec=adstock_map,
    max_lag=12,
    hill_before_adstock=True,
)
print("Model spec created.")

# 10) Fit model.
# - 20 chains x 3000 keep = 60k posterior draws after 1000 burn-in.
# - 3000 adapt steps tune the HMC step-size during burn-in.
# - target_accept_prob=0.85: tighter than HMC default (0.80) to reduce divergent transitions in the curvier corners
#   of the posterior (Hill + LogNormal can be funnel-shaped).
# - seed=0: reproducible chains across runs.
mmm = mmm_module.Meridian(input_data=input_data, model_spec=model_spec)
print("Meridian model initialized.")

mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=20,
    n_adapt=3000,
    n_burnin=1000,
    n_keep=3000,
    seed=0,
    dual_averaging_kwargs={'target_accept_prob': 0.85},
)
print("Posterior sampling completed.")

In [ ]:
# @title This is the Model Diagnostics and single html report  - takes about 3min to run (a100)
# 11) Diagnostics
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()
model_diagnostics.plot_prior_and_posterior_distribution(parameter='roi_m')
model_diagnostics.plot_prior_and_posterior_distribution(parameter='beta_m')
model_diagnostics.plot_prior_and_posterior_distribution(parameter='beta_gm')
model_diagnostics.plot_prior_and_posterior_distribution(parameter='ec_m')
visualizer.ModelFit(mmm).plot_model_fit(include_ci=True)

inference_data = mmm.inference_data
parameters_to_plot = ['roi_m', 'beta_m', 'beta_gm', 'ec_m']
az.Numba.disable_numba()
for param in parameters_to_plot:
    az.plot_trace(
        inference_data,
        var_names=param,
        compact=False,
        backend_kwargs={"constrained_layout": True},
    )
az.plot_ess(inference_data, kind='local', var_names=parameters_to_plot)
print(az.summary(inference_data, var_names=parameters_to_plot))

# 12) Summary HTML export
mmm_summarizer = summarizer.Summarizer(mmm)
start_date, end_date = '2025-06-29', '2026-06-21' #these should 52 or 53 of the week start days
out_dir = '/content/drive/MyDrive/ABC/MMM'
now = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"v13_summary_output_{start_date}_to_{end_date}_{now}.html"

mmm_summarizer.output_model_results_summary(
    filename=filename,
    filepath=out_dir,
    start_date=start_date,
    end_date=end_date,
)
print(f"Model results saved to {out_dir}/{filename}")

model_diagnostics.plot_rhat_boxplot()

In [ ]:
# @title Spend-aligned ROAS by quarter + non-media driver attribution
# ---------------------------------------------------------------------------
# WHY THIS EXISTS
# Default ROAS divides revenue *realized* in period P by spend in period P. Because media has adstock/lag
# (max_lag=12 weeks here), revenue from Q1 spend lands across Q1/Q2/Q3, making realized-period ROAS "wonky".
#
# This cell attributes every incremental-revenue dollar back to the QUARTER WHOSE MEDIA CAUSED IT. For each
# quarter Q it turns ON only the media that ran in Q (Meridian `media_selected_times`) and sums the FULL
# downstream incremental revenue across ALL later weeks (lagged tail pulled back). It also reports the old
# realized-period ROAS side-by-side so you can see the distortion the pull-back removes.
#
# ALSO IN THIS VERSION: per-quarter incremental revenue attributed to non-media TREATMENTS (Vault_Drops,
# Email_sends, StoreCount, Celebs) alongside their input values. Non-media treatments enter the model
# linearly with no adstock, so realized = aligned (no lag to pull back).
#
# NOT INCLUDED: model CONTROLS (Hurricanes, Category_Interest, LightningSales, Redemptions_*). Meridian's
# analyzer doesn't decompose them through incremental_outcome - they're absorbed into the model fit but
# aren't separately attributable here. (Possible future extension: manually compute gamma_c * X_c per
# quarter from the posterior.)
#
# CAVEAT - end of window: media in the final max_lag (12) weeks has part of its revenue tail beyond the
# data, which Meridian truncates. Quarters flagged tail_truncated=True understate true spend-aligned ROAS
# (revenue tail clipped). Earliest weeks are fine for the aligned view. Non-media has no tail issue.
# ---------------------------------------------------------------------------
import datetime
import inspect
import numpy as np
import pandas as pd
from meridian.analysis import analyzer as analyzer_module

WITH_CI   = True     # add 90% posterior credible interval on the aligned ROAS / non-media inc-rev
WRITE_CSV = True

# Aggregation per non-media treatment for the per-quarter input value. Events/sends use sum; state-like
# vars (StoreCount) use mean. Falls back to 'sum' for any variable added without an explicit choice.
non_media_agg = {
    'Vault_Drops': 'sum',
    'Email_sends': 'sum',
    'StoreCount':  'mean',
    'Celebs':      'sum'

}

analyzer = analyzer_module.Analyzer(mmm)

# safety: confirm this Meridian build supports media-execution windowing
if 'media_selected_times' not in inspect.signature(analyzer.incremental_outcome).parameters:
    raise RuntimeError(
        "This Meridian build's incremental_outcome lacks 'media_selected_times'; "
        "upgrade google-meridian to use the pulled-back ROAS method."
    )

# --- canonical weekly grid the model actually uses --------------------------
times    = pd.to_datetime(np.asarray(mmm.input_data.time.values))
q_period = pd.PeriodIndex(times, freq='Q')
quarters = sorted(q_period.unique())

# paid-media channel order (returned by incremental_outcome with include_non_paid=False)
try:
    paid_channels = list(mmm.input_data.get_all_paid_channels())
except Exception:
    paid_channels = list(media_channels)

# non-media treatments. Pulls from globals (non_media_cols from the spec cell); falls back to
# Meridian's input_data accessor if not in globals.
try:
    nm_treatments = list(non_media_cols)
except NameError:
    nm_treatments = list(getattr(mmm.input_data, 'non_media_treatments', []) or [])

n_paid = len(paid_channels)
n_nm   = len(nm_treatments)

# --- spend & impressions per channel per quarter (summed over geos) ----------
spend_map = dict(zip(media_channels, media_spend_cols))
impr_map  = dict(zip(media_channels, media_impression_cols))

_agg_cols = media_spend_cols + media_impression_cols
g = df_bq[['time'] + _agg_cols].copy()
g['time'] = pd.to_datetime(g['time'])
g['q']    = pd.PeriodIndex(g['time'], freq='Q')
gq = g.groupby('q')[_agg_cols].sum()

# --- non-media input values per quarter (per-variable aggregation) ----------
if nm_treatments:
    nm_df = df_bq[['time'] + nm_treatments].copy()
    nm_df['time'] = pd.to_datetime(nm_df['time'])
    nm_df['q']    = pd.PeriodIndex(nm_df['time'], freq='Q')
    nm_by_q = pd.DataFrame(index=sorted(nm_df['q'].unique()))
    for var in nm_treatments:
        nm_by_q[var] = nm_df.groupby('q')[var].agg(non_media_agg.get(var, 'sum'))
else:
    nm_by_q = pd.DataFrame()

# --- end-of-window truncation flag ------------------------------------------
max_lag = int(getattr(mmm.model_spec, 'max_lag', 0) or 0)
last_complete_week = times.max() - pd.Timedelta(weeks=max_lag)

rows = []
_layout_warned = False  # one-shot warning if non-paid channel layout looks off

for q in quarters:
    wk_mask   = (q_period == q)
    qweeks    = times[wk_mask]
    qlabel    = str(q)
    in_gq     = q in gq.index
    truncated = bool(qweeks.max() > last_complete_week) if max_lag else False

    # 1) SPEND-ALIGNED ("pulled back") for PAID media: only media that ran in q is ON; sum the
    #    full incremental revenue it drives across ALL weeks (lag tail included).
    inc_aligned = analyzer.incremental_outcome(
        media_selected_times=wk_mask.tolist(),
        selected_times=None,
        aggregate_geos=True,
        aggregate_times=True,
        include_non_paid_channels=False,
    ).numpy()  # (chains, draws, n_paid)

    # 2) REALIZED in q for paid + non-media in ONE call (sliced). Paid slice is the "wonky" view;
    #    non-media slice is the only view of non-media (linear, no adstock so realized = aligned).
    #    Meridian convention is paid first then non-paid; sanity-checked on first quarter below.
    inc_realized_all = analyzer.incremental_outcome(
        media_selected_times=None,
        selected_times=wk_mask.tolist(),
        aggregate_geos=True,
        aggregate_times=True,
        include_non_paid_channels=True,
    ).numpy()  # expected shape: (chains, draws, n_paid + n_nm)

    if not _layout_warned:
        _expected, _actual = n_paid + n_nm, inc_realized_all.shape[-1]
        if _actual != _expected:
            print(f"WARNING: analyzer returned {_actual} channels with non_paid=True; expected paid + "
                  f"non-media = {_expected}. Non-media attribution may be misaligned if your model has "
                  f"organic media/RF channels between the paid and non-media slices.")
        _layout_warned = True

    inc_realized = inc_realized_all[:, :, :n_paid]
    inc_nonmedia = inc_realized_all[:, :, n_paid:n_paid + n_nm] if n_nm > 0 else None

    aligned_mean  = inc_aligned.mean(axis=(0, 1))
    realized_mean = inc_realized.mean(axis=(0, 1))
    nm_mean       = inc_nonmedia.mean(axis=(0, 1)) if inc_nonmedia is not None else None
    if WITH_CI:
        aligned_lo = np.percentile(inc_aligned,  5, axis=(0, 1))
        aligned_hi = np.percentile(inc_aligned, 95, axis=(0, 1))
        nm_lo      = np.percentile(inc_nonmedia,  5, axis=(0, 1)) if inc_nonmedia is not None else None
        nm_hi      = np.percentile(inc_nonmedia, 95, axis=(0, 1)) if inc_nonmedia is not None else None

    # PAID rows
    for j, ch in enumerate(paid_channels):
        s  = float(gq.loc[q, spend_map[ch]]) if (in_gq and ch in spend_map) else 0.0
        im = float(gq.loc[q, impr_map[ch]])  if (in_gq and ch in impr_map)  else 0.0
        rec = {
            'quarter'            : qlabel,
            'channel'            : ch,
            'type'               : 'paid',
            'spend'              : s,
            'impressions'        : im,
            'input_value'        : np.nan,
            'inc_rev_aligned'    : float(aligned_mean[j]),
            'roas_aligned'       : (aligned_mean[j] / s) if s > 0 else np.nan,
            'inc_rev_realized'   : float(realized_mean[j]),
            'roas_realized_wonky': (realized_mean[j] / s) if s > 0 else np.nan,
            'eff_per_input_unit' : np.nan,
            'tail_truncated'     : truncated,
        }
        if WITH_CI:
            rec['roas_aligned_lo'] = (aligned_lo[j] / s) if s > 0 else np.nan
            rec['roas_aligned_hi'] = (aligned_hi[j] / s) if s > 0 else np.nan
        rows.append(rec)

    # NON-MEDIA rows (one per treatment, only if analyzer returned non-paid)
    if nm_mean is not None:
        for k, treat in enumerate(nm_treatments):
            val = float(nm_by_q.loc[q, treat]) if (q in nm_by_q.index and treat in nm_by_q.columns) else 0.0
            inc = float(nm_mean[k])
            rec = {
                'quarter'            : qlabel,
                'channel'            : treat,
                'type'               : 'non_media',
                'spend'              : np.nan,
                'impressions'        : np.nan,
                'input_value'        : val,
                'inc_rev_aligned'    : np.nan,   # not applicable - no adstock on non-media
                'roas_aligned'       : np.nan,
                'inc_rev_realized'   : inc,
                'roas_realized_wonky': np.nan,
                'eff_per_input_unit' : (inc / val) if val != 0 else np.nan,
                'tail_truncated'     : False,    # non-media has no adstock tail
            }
            if WITH_CI:
                rec['roas_aligned_lo'] = np.nan
                rec['roas_aligned_hi'] = np.nan
                rec['inc_rev_lo']      = float(nm_lo[k])
                rec['inc_rev_hi']      = float(nm_hi[k])
            rows.append(rec)

roas_q = pd.DataFrame(rows)

# --- blended (all-paid-channel) ROAS per quarter -----------------------------
paid_mask = roas_q['type'] == 'paid'
blended = (roas_q[paid_mask].groupby('quarter')
                  .agg(spend=('spend', 'sum'),
                       impressions=('impressions', 'sum'),
                       inc_rev_aligned=('inc_rev_aligned', 'sum'),
                       inc_rev_realized=('inc_rev_realized', 'sum'),
                       tail_truncated=('tail_truncated', 'max'))
                  .reset_index())
blended['channel']             = 'ALL_PAID (blended)'
blended['type']                = 'paid_blend'
blended['roas_aligned']        = blended['inc_rev_aligned']  / blended['spend']
blended['roas_realized_wonky'] = blended['inc_rev_realized'] / blended['spend']
roas_q = pd.concat([roas_q, blended], ignore_index=True, sort=False)
roas_q = roas_q.sort_values(['quarter', 'type', 'channel']).reset_index(drop=True)

# --- stamp the export date/time onto the data AND the filename --------------
_exported_at = datetime.datetime.now()
roas_q['exported_at'] = _exported_at.strftime('%Y-%m-%d %H:%M:%S')

# wide pivots for quick eyeballing
pivot_aligned_paid = (roas_q[roas_q['type'] == 'paid']
                        .pivot(index='quarter', columns='channel', values='roas_aligned'))
pivot_nm_revenue   = (roas_q[roas_q['type'] == 'non_media']
                        .pivot(index='quarter', columns='channel', values='inc_rev_realized'))
pivot_nm_input     = (roas_q[roas_q['type'] == 'non_media']
                        .pivot(index='quarter', columns='channel', values='input_value'))

pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
print("PAID media - spend-aligned ROAS by quarter (future incremental revenue pulled back to the spend that drove it):\n")
print(pivot_aligned_paid)
print("\nNON-MEDIA - incremental revenue attributed per quarter (linear effect, no adstock):\n")
print(pivot_nm_revenue)
print("\nNON-MEDIA - input values per quarter (sum or mean per variable, see non_media_agg):\n")
print(pivot_nm_input)
print(f"\nExport timestamp: {roas_q['exported_at'].iloc[0]}")

if WRITE_CSV:
    _dir = out_dir if 'out_dir' in globals() else '.'
    _stamp = _exported_at.strftime('%Y%m%d_%H%M%S')
    path = f"{_dir}/roas_by_quarter_spend_aligned_{_stamp}.csv"
    roas_q.to_csv(path, index=False)
    print(f"Saved: {path}")

In [ ]:
# @title Per-channel adstock decay profile (revenue by lag week)
# ---------------------------------------------------------------------------
# WHY THIS EXISTS
# Each channel has a different adstock spec (geometric for Search/Meta/PMAX/Amex,
# binomial for the Video group). This cell answers, per channel: of the
# incremental revenue a week's media eventually drives, what share lands in the
# week of impression (lag 0), the week after (lag +1), +2, ... up to max_lag.
#
# METHOD
# For each channel ch, pick a representative IMPULSE WEEK W_ch (the week ch's
# media activity peaked, restricted to >= max_lag weeks before the end of data
# so the full tail fits in-window). Then run ONE Meridian counterfactual with
# `media_selected_times` ON only at W_ch and OFF everywhere else and
# `aggregate_times=False`. The resulting incremental-outcome time series for ch
# is exactly  beta_ch * w_{ch,L} * Hill(activity_ch[W_ch])  at t = W_ch + L
# (because hill_before_adstock=True applies the saturation BEFORE the adstock
# convolution), so dividing by the sum recovers the adstock kernel directly.
# The shape is independent of spend level and of which W_ch we picked.
#
# NOTE on binomial adstock: the peak need not be at lag 0 - a "ramp" then decay
# is expected for video. half_life_wk and weeks_to_90pct handle this correctly
# because they're computed off the CUMULATIVE curve.
#
# Cost: n_channels (9) counterfactual sims. A few minutes on GPU.
# ---------------------------------------------------------------------------
import datetime
import numpy as np
import pandas as pd
from meridian.analysis import analyzer as analyzer_module

WITH_PLOT = True
WRITE_CSV = True

analyzer = analyzer_module.Analyzer(mmm)
max_lag  = int(getattr(mmm.model_spec, 'max_lag', 12) or 12)

# canonical weekly grid (the model's own time axis)
times   = pd.to_datetime(np.asarray(mmm.input_data.time.values))
n_times = len(times)

# paid channels - order matches incremental_outcome output channel dim
try:
    paid_channels = list(mmm.input_data.get_all_paid_channels())
except Exception:
    paid_channels = list(media_channels)

# channel -> media-activity column (Search = clicks, all others = impressions)
impr_map = dict(zip(media_channels, media_impression_cols))

# adstock spec per channel - for context in the printed table
try:
    adstock_spec = dict(mmm.model_spec.adstock_decay_spec or {})
except Exception:
    adstock_spec = {}

# weekly activity per channel, aligned to the model's time grid
act = df_bq[['time'] + media_impression_cols].copy()
act['time'] = pd.to_datetime(act['time'])
act = act.groupby('time')[media_impression_cols].sum().reindex(times)

# pick impulse week per channel: peak activity, restricted to leave the full
# max_lag tail inside the data window so the decay isn't truncated
safe_end = max(0, n_times - max_lag - 1)
impulse_idx, impulse_note = {}, {}
for ch in paid_channels:
    col = impr_map.get(ch)
    if col is None or col not in act.columns:
        impulse_idx[ch] = None
        impulse_note[ch] = 'no activity column mapped'
        continue
    series = act[col].fillna(0).to_numpy(dtype=float)
    safe_series = series.copy()
    if safe_end + 1 < n_times:
        safe_series[safe_end + 1:] = -1.0
    if safe_series.max() > 0:
        impulse_idx[ch]  = int(np.argmax(safe_series))
        impulse_note[ch] = 'peak activity in-window'
    elif series.max() > 0:
        impulse_idx[ch]  = int(np.argmax(series))
        impulse_note[ch] = 'peak falls in last max_lag wks - tail TRUNCATED'
    else:
        impulse_idx[ch]  = None
        impulse_note[ch] = 'no positive activity - cannot recover decay'

usable = [c for c in paid_channels if impulse_idx.get(c) is not None]
print(f"Computing per-channel decay - {len(usable)} channels x 1 counterfactual "
      f"sim each. A few minutes on GPU.")

# --- run one impulse counterfactual per channel ------------------------------
decay_rows = []
for ch in paid_channels:
    w_idx = impulse_idx.get(ch)
    if w_idx is None:
        continue
    mask = np.zeros(n_times, dtype=bool)
    mask[w_idx] = True

    inc = analyzer.incremental_outcome(
        media_selected_times=mask.tolist(),
        selected_times=None,
        aggregate_geos=True,
        aggregate_times=False,
        include_non_paid_channels=False,
    ).numpy()                                  # (chains, draws, n_times, n_channels)

    ch_idx   = paid_channels.index(ch)
    inc_mean = inc.mean(axis=(0, 1))[:, ch_idx]      # (n_times,) - this channel only
    end      = min(n_times, w_idx + max_lag + 1)
    tail     = inc_mean[w_idx:end]                   # values at lags 0..max_lag
    total    = float(tail.sum())
    if total <= 0:
        continue
    profile = tail / total
    cum     = np.cumsum(profile)
    for L, (frac, c) in enumerate(zip(profile, cum)):
        decay_rows.append({
            'channel'        : ch,
            'adstock'        : adstock_spec.get(ch, ''),
            'lag_weeks'      : L,
            'pct_of_revenue' : float(frac),
            'cumulative_pct' : float(c),
            'impulse_week'   : str(times[w_idx].date()),
            'impulse_note'   : impulse_note[ch],
        })

decay = pd.DataFrame(decay_rows)

# --- wide table: channels x lag ---------------------------------------------
wide = (decay.pivot(index='channel', columns='lag_weeks', values='pct_of_revenue')
             .reindex(paid_channels))
wide.columns = [f'wk+{L}' for L in wide.columns]

# --- per-channel summary: half-life, 90% lag, first-month concentration ------
def _lag_at_cum(g, thr):
    g = g.sort_values('lag_weeks')
    hit = g[g['cumulative_pct'] >= thr]
    return int(hit['lag_weeks'].iloc[0]) if len(hit) else np.nan

summary = []
for ch in paid_channels:
    g = decay[decay['channel'] == ch]
    if g.empty:
        summary.append({
            'channel': ch, 'adstock': adstock_spec.get(ch, ''),
            'pct_week_0': np.nan, 'pct_weeks_0_4': np.nan,
            'half_life_wk': np.nan, 'weeks_to_90pct': np.nan,
            'impulse_week': '', 'impulse_note': impulse_note.get(ch, ''),
        })
        continue
    s0  = g.loc[g['lag_weeks'] == 0, 'pct_of_revenue']
    s04 = g.loc[g['lag_weeks'] <= 4, 'pct_of_revenue'].sum()
    summary.append({
        'channel'        : ch,
        'adstock'        : adstock_spec.get(ch, ''),
        'pct_week_0'     : float(s0.iloc[0]) if len(s0) else np.nan,
        'pct_weeks_0_4'  : float(s04),
        'half_life_wk'   : _lag_at_cum(g, 0.50),
        'weeks_to_90pct' : _lag_at_cum(g, 0.90),
        'impulse_week'   : g['impulse_week'].iloc[0],
        'impulse_note'   : g['impulse_note'].iloc[0],
    })
summary_df = pd.DataFrame(summary).set_index('channel').reindex(paid_channels)

# --- stamp export date/time onto data + filenames ---------------------------
_exported_at = datetime.datetime.now()
decay['exported_at']      = _exported_at.strftime('%Y-%m-%d %H:%M:%S')
summary_df['exported_at'] = _exported_at.strftime('%Y-%m-%d %H:%M:%S')

pd.set_option('display.float_format', lambda v: f'{v:,.3f}')
print("\nShare of each channel's incremental revenue landing at lag L weeks:\n")
print(wide.fillna(0).to_string())
print("\nSummary - half-life, 90% lag, and concentration in first 5 weeks:\n")
print(summary_df[['adstock', 'pct_week_0', 'pct_weeks_0_4',
                  'half_life_wk', 'weeks_to_90pct', 'impulse_note']].to_string())

if WRITE_CSV:
    _dir   = out_dir if 'out_dir' in globals() else '.'
    _stamp = _exported_at.strftime('%Y%m%d_%H%M%S')
    p1 = f"{_dir}/decay_profile_by_channel_{_stamp}.csv"
    p2 = f"{_dir}/decay_summary_by_channel_{_stamp}.csv"
    decay.to_csv(p1, index=False)
    summary_df.to_csv(p2)
    print(f"\nSaved:\n  {p1}\n  {p2}")

if WITH_PLOT:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
    for ch in paid_channels:
        g = decay[decay['channel'] == ch].sort_values('lag_weeks')
        if g.empty:
            continue
        axes[0].plot(g['lag_weeks'], 100 * g['pct_of_revenue'], marker='o', label=ch)
        axes[1].plot(g['lag_weeks'], 100 * g['cumulative_pct'],  marker='o', label=ch)
    axes[0].set_title('% of channel revenue landing each lag week')
    axes[0].set_xlabel('Weeks since impression'); axes[0].set_ylabel('%')
    axes[1].set_title('Cumulative % of channel revenue by lag week')
    axes[1].set_xlabel('Weeks since impression'); axes[1].set_ylabel('%')
    axes[1].axhline(50, color='gray', linestyle=':',  linewidth=0.8)
    axes[1].axhline(90, color='gray', linestyle='--', linewidth=0.8)
    axes[1].legend(loc='lower right', fontsize=8)
    plt.tight_layout(); plt.show()


In [ ]:
# @title Scenario planner builder (takes 1.5rhrs) - writes to existing dashboard sheet on A100

# ---------------------------------------------------------------------------
# Defensive install + prereqs check: in case the Colab runtime was reset since cell 0 (MMMv13) ran.
# A reset wipes installed packages AND your fitted `mmm` model. The pip line below is a no-op if
# Meridian is already installed; the assert tells you clearly if you need to re-run cell 0 first.
# ---------------------------------------------------------------------------
!pip -q install --upgrade google-meridian[colab,scenarioplanner,and-cuda]

_missing = [n for n in ('mmm', 'credentials') if n not in globals()]
if _missing:
    raise RuntimeError(
        f"Missing required variable(s) from cell 0: {_missing}. Re-run cell 0 (MMMv13 model spec) "
        f"first - it fits the model and leaves it in scope as `mmm`. Runtime resets wipe in-memory "
        f"state, so the full ~45 min sampling step has to be redone."
    )

from IPython.display import HTML
import inspect

from meridian.schema.processors import (
    model_fit_processor,
    marketing_processor,
    budget_optimization_processor,
)
from meridian.schema.utils import date_range_bucketing
from scenarioplanner.converters import sheets
from scenarioplanner.converters.dataframe import dataframe_model_converter
from scenarioplanner import mmm_ui_proto_generator as mmm_ui_gen
from scenarioplanner.linkingapi import url_generator

print('Scenario Planner imports ready.')

# ---------------------------------------------------------------------------
# v13 UPDATE: write to the EXISTING dashboard spreadsheet instead of creating a new sheet each run. Target
# the sheet your dashboard already points at so refresh pulls the latest data automatically. The runtime
# tries Meridian's native helper first (if any version of upload_to_gsheet ever supports targeting by ID),
# then falls back to direct gspread writes to the same sheet ID.
# ---------------------------------------------------------------------------
TARGET_SPREADSHEET_ID = '101EYa2FK8BJ4u6SCC4cyDvgYuWrdnP2fSaHWRk0IfVU'

optimization_name = 'ABC FY26'  # @param {"type":"string"}
include_non_paid_channels = True  # @param {"type":"boolean"}

yearly = True       # @param {"type":"boolean"}
quarterly = False    # @param {"type":"boolean"}
monthly = False     # @param {"type":"boolean"}

min_spend_shift_ratio = 0.40  # @param {"type":"raw"}
max_spend_shift_ratio = 0.40  # @param {"type":"raw"}
use_optimal_frequency = False  # @param {"type":"boolean"}

max_frequency = 10.0          # @param {"type":"raw"}

time_breakdown_generators = []
if yearly:    time_breakdown_generators.append(date_range_bucketing.YearlyDateRangeGenerator)
if quarterly: time_breakdown_generators.append(date_range_bucketing.QuarterlyDateRangeGenerator)
if monthly:   time_breakdown_generators.append(date_range_bucketing.MonthlyDateRangeGenerator)

channel_constraints = [
    budget_optimization_processor.ChannelConstraintRel(
        channel_name=ch,
        spend_constraint_lower=min_spend_shift_ratio,
        spend_constraint_upper=max_spend_shift_ratio,
    )
    for ch in mmm.input_data.get_all_paid_channels()
]

budget_opt_spec = budget_optimization_processor.BudgetOptimizationSpec(
    start_date=None,
    end_date=None,
    optimization_name=optimization_name,
    grid_name='-'.join(optimization_name.lower().split(' ')),
    constraints=channel_constraints,
    use_optimal_frequency=use_optimal_frequency,
    max_frequency=max_frequency,
)

print('Running scenario inference - ~45 min on h1100.')
mmm_proto = mmm_ui_gen.create_mmm_ui_data_proto(
    mmm=mmm,
    specs=[
        model_fit_processor.ModelFitSpec(),
        marketing_processor.MarketingAnalysisSpec(
            media_summary_spec=marketing_processor.MediaSummarySpec(
                include_non_paid_channels=include_non_paid_channels,
            ),
        ),
        budget_opt_spec,
    ],
    time_breakdown_generators=time_breakdown_generators,
)

print('Converting to dataframes.')
dataframes = dataframe_model_converter.DataFrameModelConverter(mmm_proto)()

# ---------------------------------------------------------------------------
# UPLOAD - write to the existing target spreadsheet (not a new one each run).
# Strategy: try Meridian's native helper first if it supports targeting by ID; otherwise fall back to
# direct gspread upload. Most Meridian versions only support upload_to_gsheet(..., spreadsheet_name=...)
# which creates a NEW sheet, so the gspread fallback is the expected path.
# ---------------------------------------------------------------------------
print(f'Uploading to existing spreadsheet (id={TARGET_SPREADSHEET_ID})...')

_upload_params = set(inspect.signature(sheets.upload_to_gsheet).parameters.keys())
print(f"  sheets.upload_to_gsheet params: {sorted(_upload_params)}")
_native_id_param = next(
    (p for p in ('spreadsheet_id', 'sheet_id', 'existing_spreadsheet_id') if p in _upload_params),
    None,
)

spreadsheet = None
if _native_id_param:
    try:
        spreadsheet = sheets.upload_to_gsheet(
            dataframes, credentials,
            **{_native_id_param: TARGET_SPREADSHEET_ID},
        )
        print(f"  Updated via Meridian's native '{_native_id_param}' parameter.")
    except Exception as e:
        print(f"  Native upload via '{_native_id_param}' failed ({type(e).__name__}: {e}). Falling back to gspread.")
        spreadsheet = None

if spreadsheet is None:
    # Direct gspread upload to the target spreadsheet
    import gspread
    import pandas as pd
    import numpy as np

    try:
        gc = gspread.authorize(credentials)
    except Exception:
        # If the colab credentials object isn't directly acceptable, fall back to google.auth.default
        from google.auth import default as _default_creds
        _creds, _ = _default_creds(scopes=[
            'https://www.googleapis.com/auth/spreadsheets',
            'https://www.googleapis.com/auth/drive',
        ])
        gc = gspread.authorize(_creds)

    spreadsheet = gc.open_by_key(TARGET_SPREADSHEET_ID)
    print(f"  Opened existing spreadsheet: '{spreadsheet.title}'")

    # The converter's output could be a dict, namedtuple, dataclass, or a plain object with DataFrame
    # attributes. Normalize to a list of (name, df) pairs.
    if isinstance(dataframes, dict):
        df_pairs = list(dataframes.items())
    elif hasattr(dataframes, '_asdict'):
        df_pairs = list(dataframes._asdict().items())
    elif hasattr(dataframes, '__dataclass_fields__'):
        df_pairs = [(f, getattr(dataframes, f)) for f in dataframes.__dataclass_fields__]
    else:
        df_pairs = [(k, v) for k, v in vars(dataframes).items() if hasattr(v, 'columns')]

    # -----------------------------------------------------------------------
    # AUGMENT: append a "Last 52 Weeks" row to ModelDiagnostics so the dashboard surfaces both the
    # full-window fit and the recent-year fit side-by-side. Meridian's native ModelFitSpec only reports
    # one row ("All Data") computed across the full ~174-week training window - that pulls down R² to
    # ~0.71 because many video partners weren't active in 2023/early-2024. The same model fits the most
    # recent 52 weeks at ~0.82 (per the HTML summary), which is what budget decisions actually rely on.
    # We compute the recent-window metrics here from the ModelFit weekly actuals/predictions using the
    # classical formulas (Meridian's R² is Bayesian R², slightly different methodology but typically
    # within ~1-2 pts of classical for a well-fit model). NOTE: this augmentation only applies on the
    # gspread fallback path - the native path uploads the raw `dataframes` object before we touch it.
    # -----------------------------------------------------------------------
    def _augment_diagnostics_with_recent_window(df_pairs, n_weeks=52):
        pairs = dict(df_pairs)
        if 'ModelDiagnostics' not in pairs or 'ModelFit' not in pairs:
            print(f"  (skip recent-window diagnostics: ModelDiagnostics/ModelFit tab not present)")
            return df_pairs

        fit = pairs['ModelFit'].copy()

        # Flexible column matching - Meridian's column names vary across versions.
        def _find_col(df, candidates):
            lookup = {c.lower(): c for c in df.columns}
            for cand in candidates:
                if cand in lookup:
                    return lookup[cand]
            return None

        time_col   = _find_col(fit, ['time', 'date', 'week', 'time_period', 'period'])
        actual_col = _find_col(fit, ['actual', 'actual_kpi', 'actual_revenue', 'realized', 'observed', 'y_true'])
        pred_col   = _find_col(fit, ['expected', 'predicted', 'expected_kpi', 'expected_revenue',
                                     'mean', 'prediction', 'fit', 'y_pred'])
        if time_col is None or actual_col is None or pred_col is None:
            print(f"  (skip recent-window diagnostics: couldn't identify time/actual/predicted cols in "
                  f"ModelFit; columns are {list(fit.columns)})")
            return df_pairs

        # Sort by time, take last N weeks
        fit[time_col] = pd.to_datetime(fit[time_col], errors='coerce')
        fit = fit.dropna(subset=[time_col, actual_col, pred_col]).sort_values(time_col)
        recent = fit.tail(n_weeks) if len(fit) >= n_weeks else fit
        actual = recent[actual_col].astype(float).to_numpy()
        pred   = recent[pred_col].astype(float).to_numpy()

        # Classical R² = 1 - SSE/SST; MAPE = mean(|err|/|actual|); wMAPE = sum|err|/sum|actual|.
        sse = float(np.sum((actual - pred) ** 2))
        sst = float(np.sum((actual - np.mean(actual)) ** 2))
        r_squared = 1.0 - (sse / sst) if sst > 0 else np.nan
        abs_err = np.abs(actual - pred)
        with np.errstate(divide='ignore', invalid='ignore'):
            mape_terms = np.where(actual != 0, abs_err / np.abs(actual), np.nan)
        mape  = float(np.nanmean(mape_terms))
        wmape = float(np.sum(abs_err) / np.sum(np.abs(actual))) if np.sum(np.abs(actual)) > 0 else np.nan

        # Build the new row matching existing column names (defensive on naming).
        diag = pairs['ModelDiagnostics'].copy()
        new_row = {col: '' for col in diag.columns}
        for col in diag.columns:
            cl = col.lower().strip()
            if 'dataset' in cl or cl in ('name', 'window'):
                new_row[col] = f'Last {n_weeks} Weeks'
            elif 'r' in cl and 'square' in cl:
                new_row[col] = r_squared
            elif cl == 'mape':
                new_row[col] = mape
            elif cl == 'wmape':
                new_row[col] = wmape

        pairs['ModelDiagnostics'] = pd.concat([diag, pd.DataFrame([new_row])], ignore_index=True)
        print(f"  Added 'Last {n_weeks} Weeks' row to ModelDiagnostics: "
              f"R²={r_squared:.4f}, MAPE={mape:.4f}, wMAPE={wmape:.4f} "
              f"(covering {recent[time_col].min().date()} -> {recent[time_col].max().date()})")
        return list(pairs.items())

    df_pairs = _augment_diagnostics_with_recent_window(df_pairs, n_weeks=52)

    # -----------------------------------------------------------------------
    # AUGMENT: build a sibling MediaROI_Aligned tab with spend-aligned (pulled-back) ROI per channel
    # per analysis period. The native MediaROI tab uses realized-period attribution: revenue *landing
    # in period P* / spend *in period P*. With max_lag=12 and binomial adstock spreading video revenue
    # across 10+ weeks, that's systematically wrong - especially for video where only ~14% of revenue
    # lands in the impression week (vs ~52% for geometric channels). The aligned tab attributes the
    # FULL lagged tail back to the spend that drove it, using the same `media_selected_times` trick as
    # the standalone spend-aligned ROAS-by-quarter cell. Schema matches MediaROI exactly so Looker
    # Studio data sources can be repointed with one click. Roughly 2-5 min cost (one incremental_outcome
    # + one marginal_roi call per analysis period; ~20-30 periods typical).
    # -----------------------------------------------------------------------
    from meridian.analysis import analyzer as _analyzer_module
    _analyzer  = _analyzer_module.Analyzer(mmm)
    _times     = pd.to_datetime(np.asarray(mmm.input_data.time.values))
    _paid_chs  = list(mmm.input_data.get_all_paid_channels())
    _spend_map = dict(zip(media_channels, media_spend_cols))

    def _build_media_roi_aligned(df_pairs):
        pairs = dict(df_pairs)
        if 'MediaROI' not in pairs:
            print(f"  (skip MediaROI_Aligned: MediaROI tab not present)")
            return df_pairs

        src = pairs['MediaROI'].copy()
        period_cols = ['Analysis Period', 'Analysis Date Start', 'Analysis Date End']
        if not all(c in src.columns for c in period_cols):
            print(f"  (skip MediaROI_Aligned: missing period cols. MediaROI columns: {list(src.columns)})")
            return df_pairs

        periods = src[period_cols].drop_duplicates().reset_index(drop=True)
        n_paid = len(_paid_chs)
        print(f"  Computing MediaROI_Aligned for {len(periods)} periods x {n_paid} channels...")

        new_rows = []
        for _, period in periods.iterrows():
            pname = period['Analysis Period']
            try:
                start = pd.to_datetime(period['Analysis Date Start'])
                end   = pd.to_datetime(period['Analysis Date End'])
            except Exception as e:
                print(f"    [{pname}] skip - bad date: {e}")
                continue

            wk_mask = (_times >= start) & (_times < end)
            if not wk_mask.any():
                print(f"    [{pname}] skip - no model weeks in {start.date()} -> {end.date()}")
                continue

            # Aligned incremental outcome: media ON only in period weeks, sum revenue over ALL weeks.
            try:
                inc_aligned = _analyzer.incremental_outcome(
                    media_selected_times=wk_mask.tolist(),
                    selected_times=None,
                    aggregate_geos=True,
                    aggregate_times=True,
                    include_non_paid_channels=False,
                ).numpy()  # (chains, draws, n_paid)
            except Exception as e:
                print(f"    [{pname}] skip - incremental_outcome failed: {type(e).__name__}: {e}")
                continue

            # Spend-aligned marginal ROI: Meridian's marginal_roi(selected_times=...) already uses
            # pulled-back semantics (it captures the full lagged response to a perturbation in spend
            # within the selected window). Fall back to NaN if the method isn't available or fails.
            mroi_mean = np.full(n_paid, np.nan)
            try:
                mroi_tensor = _analyzer.marginal_roi(
                    selected_times=wk_mask.tolist(),
                    aggregate_geos=True,
                    use_posterior=True,
                ).numpy()
                mroi_mean = mroi_tensor.mean(axis=(0, 1))
            except Exception as e:
                print(f"    [{pname}] marginal_roi failed ({type(e).__name__}: {e}); setting to NaN")

            aligned_mean = inc_aligned.mean(axis=(0, 1))
            aligned_lo   = np.percentile(inc_aligned,  5, axis=(0, 1))
            aligned_hi   = np.percentile(inc_aligned, 95, axis=(0, 1))

            period_df = df_bq[(df_bq['time'] >= start) & (df_bq['time'] < end)]

            for j, ch in enumerate(_paid_chs):
                spend_col = _spend_map.get(ch)
                spend_val = float(period_df[spend_col].sum()) if (spend_col in period_df.columns) else 0.0
                if spend_val > 0:
                    roi    = float(aligned_mean[j]) / spend_val
                    roi_lo = float(aligned_lo[j])   / spend_val
                    roi_hi = float(aligned_hi[j])   / spend_val
                else:
                    roi = roi_lo = roi_hi = np.nan

                # Preserve Effectiveness from MediaROI (it's just beta_m, unaffected by aligned vs
                # realized split - same posterior value either way).
                eff = ''
                match = src[(src['Channel'] == ch) & (src['Analysis Period'] == pname)]
                if not match.empty and 'Effectiveness' in match.columns:
                    eff = match['Effectiveness'].iloc[0]

                new_rows.append({
                    'Channel': ch,
                    'Spend': spend_val,
                    'Effectiveness': eff,
                    'ROI': roi,
                    'ROI CI Low': roi_lo,
                    'ROI CI High': roi_hi,
                    'Marginal ROI': float(mroi_mean[j]) if np.isfinite(mroi_mean[j]) else np.nan,
                    'Is Revenue KPI': True,
                    'Analysis Period': pname,
                    'Analysis Date Start': period['Analysis Date Start'],
                    'Analysis Date End': period['Analysis Date End'],
                })

        aligned_df = pd.DataFrame(new_rows)
        # Preserve MediaROI column ordering so tabs are visually parallel.
        ordered_cols = [c for c in src.columns if c in aligned_df.columns]
        aligned_df = aligned_df[ordered_cols + [c for c in aligned_df.columns if c not in ordered_cols]]
        pairs['MediaROI_Aligned'] = aligned_df
        print(f"  Built MediaROI_Aligned: {len(aligned_df)} rows x {len(aligned_df.columns)} cols")
        return list(pairs.items())

    df_pairs = _build_media_roi_aligned(df_pairs)

    print(f"  Writing {len(df_pairs)} dataframes to worksheets:")

    for sheet_name, df in df_pairs:
        # Google Sheets worksheet titles: max 100 chars, no '/' or '\' (and a few others); be defensive.
        safe_name = str(sheet_name).strip().replace('/', '_').replace('\\', '_')[:100]
        if not safe_name:
            safe_name = 'sheet_unnamed'

        # Find or create the worksheet
        try:
            ws = spreadsheet.worksheet(safe_name)
            ws.clear()
        except gspread.exceptions.WorksheetNotFound:
            ws = spreadsheet.add_worksheet(
                title=safe_name,
                rows=max(len(df) + 10, 100),
                cols=max(len(df.columns) + 5, 26),
            )

        # Convert DataFrame to gspread-friendly values: NaN/inf -> '', datetimes -> ISO strings, numerics
        # preserved as numbers (so Looker Studio sees them as numeric). Stringifying everything would
        # break any numeric formulas/aggregations downstream. NOTE: pandas .notna() treats inf as valid
        # (only catches NaN/None), but the JSON encoder rejects inf outright with "Out of range float
        # values are not JSON compliant". Scenario planner outputs produce inf wherever there's a
        # divide-by-zero (marginal ROI at zero spend, eff_per_input_unit when input=0, etc.), so we
        # replace inf/-inf with NaN first and then funnel the NaNs to ''.
        df_copy = df.copy()
        for _col in df_copy.select_dtypes(include=['datetime64', 'datetimetz']).columns:
            df_copy[_col] = df_copy[_col].dt.strftime('%Y-%m-%d %H:%M:%S')
        df_copy = df_copy.replace([np.inf, -np.inf], np.nan)
        df_copy = df_copy.astype(object).where(df_copy.notna(), '')

        values = [df_copy.columns.astype(str).tolist()] + df_copy.values.tolist()
        ws.update(range_name='A1', values=values)
        print(f"    - {safe_name}: {len(df)} rows x {len(df.columns)} cols")

print(f'\nDone. Spreadsheet URL: https://docs.google.com/spreadsheets/d/{TARGET_SPREADSHEET_ID}/edit')

# ---------------------------------------------------------------------------
# Scenario Planner exporter to Looker Studio (best-effort). url_generator expects Meridian's own
# spreadsheet object; if we went through the gspread fallback, it likely won't accept the gspread
# Spreadsheet object. Your existing dashboard pointed at this sheet will refresh automatically anyway.
# ---------------------------------------------------------------------------
try:
    report_url = url_generator.create_report_url(spreadsheet)
    HTML(f'<a href="{report_url}" target="_blank">Open ABC MMM Scenario Planner in Looker Studio</a>')
except Exception as e:
    print(f"\nLooker Studio URL not auto-generated (expected when using the gspread fallback path):")
    print(f"  {type(e).__name__}: {e}")
    print(f"Your existing dashboard pointed at this spreadsheet should refresh with the new data automatically.")
